# AirSense — Perhitungan Indeks Standar Pencemar Udara (ISPU)

Notebook ini digunakan untuk mengembangkan dan memvalidasi proses perhitungan Indeks Standar Pencemar Udara (ISPU) berdasarkan Permen LHK No. 14 Tahun 2020.

### Tujuan
1. Menyiapkan konsentrasi parameter untuk perhitungan ISPU.
2. Mengimplementasikan breakpoint ISPU berdasarkan regulasi.
3. Menghitung indeks masing-masing parameter menggunakan interpolasi linear.
4. Menentukan ISPU dominan dan kategori kualitas udara.
5. Memvalidasi hasil perhitungan sebelum logika dipindahkan ke pipeline AirSense.

> Perhitungan ISPU merupakan proses berbasis regulasi (rule-based), bukan model machine learning.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print("Library berhasil di-import.")

Library berhasil di-import.


In [11]:
DATA_PATH = Path("../../dummy_data/output/dummy_tb_konsentrasi_gas.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

df["created_at"] = pd.to_datetime(
    df["created_at"],
    utc=True,
    errors="coerce"
)

df = df.sort_values("created_at").reset_index(drop=True)

print("Jumlah data :", len(df))
print("Awal        :", df["created_at"].min())
print("Akhir       :", df["created_at"].max())

df.head()

Jumlah data : 10081
Awal        : 2026-08-30 13:49:00+00:00
Akhir       : 2026-09-06 13:49:00+00:00


,created_at,pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,temperature,humidity
0,2026-08-30 13:49:00+00:00,11.31,14.87,2682.45,0.42,20.61,38.99,48.40
1,2026-08-30 13:50:00+00:00,16.18,21.71,2673.63,0.00,73.92,36.95,52.98
2,2026-08-30 13:51:00+00:00,15.33,17.59,2818.74,0.00,21.04,37.30,52.37
3,2026-08-30 13:52:00+00:00,13.20,9.97,2548.78,0.43,21.05,38.50,49.88
4,2026-08-30 13:53:00+00:00,16.70,17.53,2537.62,0.00,80.64,38.24,53.81


## Breakpoint ISPU

Breakpoint konsentrasi mengacu pada Lampiran I Permen LHK No. 14 Tahun 2020. Parameter konsentrasi dinyatakan dalam µg/m³.

Perhitungan indeks dilakukan menggunakan interpolasi linear antara batas bawah dan batas atas konsentrasi pada masing-masing kategori ISPU.

In [12]:
#Simpan Breakpoint ISPU
ISPU_BREAKPOINTS = {
    "pm25_ugm3": [
        (0, 15.5, 0, 50),
        (15.5, 55.4, 50, 100),
        (55.4, 150.4, 100, 200),
        (150.4, 250.4, 200, 300),
        (250.4, 500.0, 300, 500),
    ],

    "pm10_ugm3": [
        (0, 50, 0, 50),
        (50, 150, 50, 100),
        (150, 350, 100, 200),
        (350, 420, 200, 300),
        (420, 500, 300, 500),
    ],

    "co_ugm3": [
        (0, 4000, 0, 50),
        (4000, 8000, 50, 100),
        (8000, 15000, 100, 200),
        (15000, 30000, 200, 300),
        (30000, 45000, 300, 500),
    ],

    "o3_ugm3": [
        (0, 120, 0, 50),
        (120, 235, 50, 100),
        (235, 400, 100, 200),
        (400, 800, 200, 300),
        (800, 1000, 300, 500),
    ],

    "no2_ugm3": [
        (0, 80, 0, 50),
        (80, 200, 50, 100),
        (200, 1130, 100, 200),
        (1130, 2260, 200, 300),
        (2260, 3000, 300, 500),
    ]
}

In [13]:
#Buat Fungsi Interpolasi
def calculate_ispu(concentration, breakpoints):
    if pd.isna(concentration):
        return np.nan

    if concentration < 0:
        return np.nan

    for xb, xa, ib, ia in breakpoints:
        if concentration <= xa:
            ispu = (
                ((ia - ib) / (xa - xb))
                * (concentration - xb)
                + ib
            )
            return round(ispu)

    # Konsentrasi melebihi breakpoint terakhir
    xb, xa, ib, ia = breakpoints[-1]

    ispu = (
        ((ia - ib) / (xa - xb))
        * (concentration - xb)
        + ib
    )

    return round(ispu)

In [14]:
#Uji Fungsi
test_values = [
    ("PM2.5", 15.5, ISPU_BREAKPOINTS["pm25_ugm3"]),
    ("PM2.5", 55.4, ISPU_BREAKPOINTS["pm25_ugm3"]),
    ("PM10", 50, ISPU_BREAKPOINTS["pm10_ugm3"]),
    ("CO", 4000, ISPU_BREAKPOINTS["co_ugm3"]),
    ("O3", 120, ISPU_BREAKPOINTS["o3_ugm3"]),
    ("NO2", 80, ISPU_BREAKPOINTS["no2_ugm3"]),
]

for parameter, concentration, bp in test_values:
    result = calculate_ispu(concentration, bp)

    print(
        f"{parameter:6} | "
        f"Konsentrasi = {concentration:8.1f} | "
        f"ISPU = {result}"
    )

PM2.5  | Konsentrasi =     15.5 | ISPU = 50
PM2.5  | Konsentrasi =     55.4 | ISPU = 100
PM10   | Konsentrasi =     50.0 | ISPU = 50
CO     | Konsentrasi =   4000.0 | ISPU = 50
O3     | Konsentrasi =    120.0 | ISPU = 50
NO2    | Konsentrasi =     80.0 | ISPU = 50


## 4. Persiapan Konsentrasi untuk Perhitungan ISPU

Data AirSense memiliki resolusi pengukuran satu menit. Sebelum dikonversi menjadi indeks, data konsentrasi perlu dipersiapkan sesuai periode pengukuran yang digunakan dalam perhitungan ISPU.

Pada tahap pengembangan ini digunakan rolling aggregation sehingga nilai indeks pada suatu timestamp dihitung berdasarkan data historis yang tersedia sebelum timestamp tersebut. Pendekatan ini juga menghindari penggunaan data masa depan (future leakage).

In [15]:
df_ispu = df.copy()

df_ispu = (
    df_ispu
    .set_index("created_at")
    .sort_index()
)

df_ispu.head()

,pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,temperature,humidity
created_at,,,,,,,
2026-08-30 13:49:00+00:00,11.31,14.87,2682.45,0.42,20.61,38.99,48.40
2026-08-30 13:50:00+00:00,16.18,21.71,2673.63,0.00,73.92,36.95,52.98
2026-08-30 13:51:00+00:00,15.33,17.59,2818.74,0.00,21.04,37.30,52.37
2026-08-30 13:52:00+00:00,13.20,9.97,2548.78,0.43,21.05,38.50,49.88
2026-08-30 13:53:00+00:00,16.70,17.53,2537.62,0.00,80.64,38.24,53.81


In [16]:
#Rolling 24 jam
pollutant_cols = [
    "pm25_ugm3",
    "pm10_ugm3",
    "co_ugm3",
    "no2_ugm3",
    "o3_ugm3"
]

for col in pollutant_cols:
    df_ispu[f"{col}_24h"] = (
        df_ispu[col]
        .rolling("24h", min_periods=1440)
        .mean()
    )

df_ispu[
    pollutant_cols +
    [f"{col}_24h" for col in pollutant_cols]
].tail()

,pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,pm25_ugm3_24h,pm10_ugm3_24h,co_ugm3_24h,no2_ugm3_24h,o3_ugm3_24h
created_at,,,,,,,,,,
2026-09-06 13:45:00+00:00,19.65,25.29,2581.13,0.00,20.26,16.40,18.75,2958.72,0.15,26.28
2026-09-06 13:46:00+00:00,15.32,22.09,2616.96,0.00,59.74,16.39,18.75,2958.68,0.15,26.31
2026-09-06 13:47:00+00:00,14.47,20.78,2823.31,0.00,64.49,16.39,18.75,2958.77,0.15,26.30
2026-09-06 13:48:00+00:00,16.02,17.24,2803.42,0.00,20.95,16.39,18.75,2958.90,0.15,26.30
2026-09-06 13:49:00+00:00,15.61,17.04,2717.81,0.47,19.65,16.39,18.75,2958.78,0.15,26.26


In [17]:
rolling_cols = [
    f"{col}_24h"
    for col in pollutant_cols
]

print("Total data:", len(df_ispu))
print("\nJumlah nilai rolling 24 jam yang tersedia:")
print(df_ispu[rolling_cols].notna().sum())

print("\nTimestamp pertama yang memiliki rolling 24 jam lengkap:")

complete_24h = df_ispu[rolling_cols].notna().all(axis=1)

print(
    df_ispu.index[complete_24h][0]
    if complete_24h.any()
    else "Belum tersedia"
)

Total data: 10081

Jumlah nilai rolling 24 jam yang tersedia:
pm25_ugm3_24h    8642
pm10_ugm3_24h    8642
co_ugm3_24h      8642
no2_ugm3_24h     8642
o3_ugm3_24h      8642
dtype: int64

Timestamp pertama yang memiliki rolling 24 jam lengkap:
2026-08-31 13:48:00+00:00


In [18]:
#Hitung ISPU lima Parameter
ispu_mapping = {
    "pm25_ugm3": "pm25_ispu",
    "pm10_ugm3": "pm10_ispu",
    "co_ugm3": "co_ispu",
    "no2_ugm3": "no2_ispu",
    "o3_ugm3": "o3_ispu"
}

for pollutant, ispu_col in ispu_mapping.items():
    rolling_col = f"{pollutant}_24h"

    df_ispu[ispu_col] = df_ispu[rolling_col].apply(
        lambda x: calculate_ispu(
            x,
            ISPU_BREAKPOINTS[pollutant]
        )
    )

In [19]:
ispu_cols = list(ispu_mapping.values())

df_ispu[
    rolling_cols + ispu_cols
].dropna().head(10)

,pm25_ugm3_24h,pm10_ugm3_24h,co_ugm3_24h,no2_ugm3_24h,o3_ugm3_24h,pm25_ispu,pm10_ispu,co_ispu,no2_ispu,o3_ispu
created_at,,,,,,,,,,
2026-08-31 13:48:00+00:00,15.29,17.51,2921.19,3.12,26.51,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:49:00+00:00,15.29,17.51,2921.19,3.12,26.51,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:50:00+00:00,15.29,17.50,2921.26,3.12,26.51,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:51:00+00:00,15.29,17.51,2921.16,3.12,26.56,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:52:00+00:00,15.29,17.51,2921.20,3.12,26.56,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:53:00+00:00,15.29,17.51,2921.44,3.12,26.52,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:54:00+00:00,15.29,17.50,2921.11,3.12,26.55,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:55:00+00:00,15.29,17.50,2921.07,3.12,26.51,49.00,18.00,37.00,2.00,11.00
2026-08-31 13:56:00+00:00,15.29,17.50,2921.16,3.12,26.46,49.00,18.00,37.00,2.00,11.00


In [20]:
df_ispu[ispu_cols].describe().round(2)

,pm25_ispu,pm10_ispu,co_ispu,no2_ispu,o3_ispu
count,8642.00,8642.00,8642.00,8642.00,8642.00
mean,50.18,18.16,37.13,0.36,11.00
std,0.93,0.84,0.41,0.59,0.00
min,49.00,17.00,36.00,0.00,11.00
25%,49.00,17.00,37.00,0.00,11.00
50%,51.00,18.00,37.00,0.00,11.00
75%,51.00,19.00,37.00,1.00,11.00
max,52.00,20.00,38.00,2.00,11.00


In [21]:
#Menentukan ISPU Maksimum
valid_ispu = df_ispu[ispu_cols].notna().all(axis=1)

df_ispu["ispu_total"] = np.nan
df_ispu["dominant_pollutant"] = pd.NA

df_ispu.loc[valid_ispu, "ispu_total"] = (
    df_ispu.loc[valid_ispu, ispu_cols]
    .max(axis=1)
)

df_ispu.loc[valid_ispu, "dominant_pollutant"] = (
    df_ispu.loc[valid_ispu, ispu_cols]
    .idxmax(axis=1)
    .str.replace("_ispu", "", regex=False)
    .str.upper()
)

In [22]:
#Kategori resmi
def ispu_category(value):
    if pd.isna(value):
        return pd.NA
    elif value <= 50:
        return "Baik"
    elif value <= 100:
        return "Sedang"
    elif value <= 200:
        return "Tidak Sehat"
    elif value <= 300:
        return "Sangat Tidak Sehat"
    else:
        return "Berbahaya"

In [23]:
#Penerapan
df_ispu["ispu_category"] = (
    df_ispu["ispu_total"]
    .apply(ispu_category)
)

In [24]:
#Hasil Akhir
result_cols = [
    "pm25_ugm3_24h",
    "pm10_ugm3_24h",
    "co_ugm3_24h",
    "no2_ugm3_24h",
    "o3_ugm3_24h",
    "pm25_ispu",
    "pm10_ispu",
    "co_ispu",
    "no2_ispu",
    "o3_ispu",
    "ispu_total",
    "dominant_pollutant",
    "ispu_category"
]

df_ispu[result_cols].dropna().tail(10)

,pm25_ugm3_24h,pm10_ugm3_24h,co_ugm3_24h,no2_ugm3_24h,o3_ugm3_24h,pm25_ispu,pm10_ispu,co_ispu,no2_ispu,o3_ispu,ispu_total,dominant_pollutant,ispu_category
created_at,,,,,,,,,,,,,
2026-09-06 13:40:00+00:00,16.39,18.75,2958.53,0.15,26.25,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:41:00+00:00,16.39,18.75,2958.51,0.15,26.25,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:42:00+00:00,16.39,18.75,2958.66,0.15,26.25,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:43:00+00:00,16.40,18.75,2958.75,0.15,26.25,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:44:00+00:00,16.40,18.75,2959.01,0.15,26.28,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:45:00+00:00,16.40,18.75,2958.72,0.15,26.28,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:46:00+00:00,16.39,18.75,2958.68,0.15,26.31,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:47:00+00:00,16.39,18.75,2958.77,0.15,26.30,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang
2026-09-06 13:48:00+00:00,16.39,18.75,2958.90,0.15,26.30,51.00,19.00,37.00,0.00,11.00,51.00,PM25,Sedang


In [25]:
#Tiga Pemeriksaan
print("Distribusi kategori ISPU:")
print(df_ispu["ispu_category"].value_counts(dropna=False))

print("\nParameter pencemar dominan:")
print(df_ispu["dominant_pollutant"].value_counts(dropna=False))

print("\nISPU maksimum:")
print(df_ispu["ispu_total"].max())

Distribusi kategori ISPU:
ispu_category
Sedang    4515
Baik      4127
<NA>      1439
Name: count, dtype: int64

Parameter pencemar dominan:
dominant_pollutant
PM25    8642
<NA>    1439
Name: count, dtype: int64

ISPU maksimum:
52.0


### Interpretasi Hasil Perhitungan ISPU

Dari 10.081 observasi, sebanyak 8.642 observasi memiliki window 24 jam lengkap dan dapat digunakan pada perhitungan ISPU. Sebanyak 1.439 observasi awal belum memiliki riwayat 24 jam yang lengkap sehingga tidak diberikan nilai ISPU.

Pada dataset dummy, nilai ISPU total berada pada rentang kategori Baik hingga Sedang dengan nilai maksimum sebesar 52. PM2.5 menjadi parameter dominan pada seluruh observasi yang memiliki window lengkap.

Hasil ini menunjukkan bahwa penggunaan rolling mean 24 jam menghasilkan indeks yang relatif stabil karena perubahan atau spike jangka pendek pada data konsentrasi menjadi lebih halus setelah proses agregasi.

Karena data yang digunakan merupakan data dummy, distribusi kategori dan parameter dominan ini tidak dapat dianggap sebagai gambaran kondisi kualitas udara sebenarnya. Hasil akhir AirSense perlu dihitung kembali menggunakan data riil dari perangkat IoT.

In [26]:
#Melakukan mapping untuk menyamakan penamaan
display_names = {
    "pm25_ispu": "PM2.5",
    "pm10_ispu": "PM10",
    "co_ispu": "CO",
    "no2_ispu": "NO₂",
    "o3_ispu": "O₃"
}

df_ispu.loc[valid_ispu, "dominant_pollutant"] = (
    df_ispu.loc[valid_ispu, ispu_cols]
    .idxmax(axis=1)
    .map(display_names)
)

In [27]:
df_ispu["dominant_pollutant"].value_counts(dropna=False)

dominant_pollutant
PM2.5    8642
<NA>     1439
Name: count, dtype: int64

In [28]:
#Automated Validation agar rapi
sample_concentration = 16.39

manual_ispu = (
    ((100 - 50) / (55.4 - 15.5))
    * (sample_concentration - 15.5)
    + 50
)

function_ispu = calculate_ispu(
    sample_concentration,
    ISPU_BREAKPOINTS["pm25_ugm3"]
)

print("Perhitungan manual :", manual_ispu)
print("Hasil fungsi        :", function_ispu)
print(
    "Validasi            :",
    "PASS" if round(manual_ispu) == function_ispu else "FAIL"
)

Perhitungan manual : 51.11528822055138
Hasil fungsi        : 51
Validasi            : PASS
